In [1]:
import pandas as pd

# Carichiamo l'anagrafica
df_patients = pd.read_csv('patients.csv')

# Visualizziamo tutte le colonne disponibili e le prime 5 righe
pd.set_option('display.max_columns', None)
print(df_patients.head())

                                     Id   BIRTHDATE DEATHDATE          SSN  \
0  b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85  2019-02-17       NaN  999-65-3251   
1  c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8  2005-07-04       NaN  999-49-3323   
2  339144f8-50e1-633e-a013-f361391c4cff  1998-05-11       NaN  999-10-8743   
3  d488232e-bf14-4bed-08c0-a82f34b6a197  2003-01-28       NaN  999-56-6057   
4  217f95a3-4e10-bd5d-fb67-0cfb5e8ba075  1993-12-23       NaN  999-91-4320   

     DRIVERS    PASSPORT PREFIX       FIRST            LAST SUFFIX MAIDEN  \
0        NaN         NaN    NaN    Damon455      Langosh790    NaN    NaN   
1  S99941126         NaN    NaN       Thi53       Wunsch504    NaN    NaN   
2  S99996708  X75063318X    Mr.      Chi716  Greenfelder433    NaN    NaN   
3  S99929424         NaN    Ms.  Phillis443       Walter473    NaN    NaN   
4  S99991143  X44132498X    Mr.  Jerrold404       Herzog843    NaN    NaN   

  MARITAL   RACE    ETHNICITY GENDER                        BIRTHPLA

In [2]:
import hashlib
import pandas as pd # (Assumendo che pandas sia già importato come pd)

# 1. HASHING (Pseudonimizzazione)
# Creiamo una funzione per calcolare l'hash SHA-256 del SSN
def hash_ssn(ssn):
    if pd.isna(ssn):
        return ssn
    # Convertiamo in stringa, codifichiamo e calcoliamo l'hash crittografico
    return hashlib.sha256(str(ssn).encode('utf-8')).hexdigest()

# Applichiamo la funzione creando una nuova colonna sicura
df_patients['SSN_HASHED'] = df_patients['SSN'].apply(hash_ssn)


# 2. MASKING (Mascheramento)
# Creiamo una funzione che mantiene solo l'iniziale di un nome/cognome
def mask_string(testo):
    if pd.isna(testo):
        return testo
    testo = str(testo)
    return testo[0] + "***" if len(testo) > 0 else testo

# Applichiamo il mascheramento ai nomi
df_patients['FIRST_MASKED'] = df_patients['FIRST'].apply(mask_string)
df_patients['LAST_MASKED'] = df_patients['LAST'].apply(mask_string)


# 3. SOPPRESSIONE (Rimozione degli identificatori diretti in chiaro)
# Definiamo la lista delle colonne troppo specifiche o non utili all'analisi
colonne_da_rimuovere = [
    'SSN', 'FIRST', 'LAST', 'ADDRESS',
    'DRIVERS', 'PASSPORT', 'PREFIX', 'SUFFIX',
    'MAIDEN', 'LAT', 'LON'
]

# Rimuoviamo le colonne dal dataframe
df_patients = df_patients.drop(columns=colonne_da_rimuovere)

# Visualizziamo il risultato con le nuove colonne
print(df_patients[['Id', 'SSN_HASHED', 'FIRST_MASKED', 'LAST_MASKED', 'CITY', 'ZIP']].head())

                                     Id  \
0  b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85   
1  c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8   
2  339144f8-50e1-633e-a013-f361391c4cff   
3  d488232e-bf14-4bed-08c0-a82f34b6a197   
4  217f95a3-4e10-bd5d-fb67-0cfb5e8ba075   

                                          SSN_HASHED FIRST_MASKED LAST_MASKED  \
0  78fbbf667c1c3462734def2dd884a62368f162446b735d...         D***        L***   
1  ec865647ae155fd9d1a2ef188a3736c69117acf6db18c3...         T***        W***   
2  e285dabc7af8903e3f225696d758466c36c685647808e6...         C***        G***   
3  a35fa26ce3d516d639b58af1fc513453d973c52b2aab82...         P***        W***   
4  4828a69fd838a399f5c049cf71c4bb6641ad304d19bc4f...         J***        H***   

          CITY     ZIP  
0  Springfield  1104.0  
1   Bellingham     NaN  
2       Boston  2131.0  
3      Hingham  2043.0  
4       Revere     NaN  


In [3]:
import datetime
import pandas as pd

# 1. Calcolo dell'Età e Generalizzazione (Fasce di 20 anni)
# Assicuriamoci che BIRTHDATE sia nel formato corretto
df_patients['BIRTHDATE'] = pd.to_datetime(df_patients['BIRTHDATE'], errors='coerce')
anno_corrente = datetime.datetime.now().year
df_patients['AGE'] = anno_corrente - df_patients['BIRTHDATE'].dt.year

# Funzione per raggruppare l'età in ventenni (es. 34 diventa "20-39")
def generalizza_eta(eta):
    if pd.isna(eta):
        return "Sconosciuta"
    ventennio = int((eta // 20) * 20)
    return f"{ventennio}-{ventennio+19}"

df_patients['AGE_GROUP'] = df_patients['AGE'].apply(generalizza_eta)

# 2. Generalizzazione del CAP (ZIP Code a 2 cifre)
# Funzione per mantenere solo le prime 2 cifre del CAP
def generalizza_cap(zip_code):
    if pd.isna(zip_code):
        return "00***" # Gestione dei dati mancanti

    # In America i CAP possono iniziare con zero (es. 01104)
    zip_str = str(int(zip_code)).zfill(5)

    # Teniamo le prime 2 cifre e mascheriamo le ultime 3
    return zip_str[:2] + "***"

df_patients['ZIP_GROUP'] = df_patients['ZIP'].apply(generalizza_cap)

# 3. Calcolo della K-Anonymity
# I nostri quasi-identificatori sono la fascia d'età, il sesso e il CAP generico
quasi_identifiers = ['AGE_GROUP', 'GENDER', 'ZIP_GROUP']

# Raggruppiamo i pazienti che hanno la stessa identica combinazione
k_anon_counts = df_patients.groupby(quasi_identifiers).size().reset_index(name='Count')

# Ordiniamo per vedere i gruppi più piccoli (quelli più a rischio)
k_anon_counts = k_anon_counts.sort_values(by='Count')

print("=== LE CLASSI DI EQUIVALENZA PIÙ PICCOLE (AGGIORNATE) ===")
print(k_anon_counts.head(10))

# Il k-anonymity del dataset è il gruppo più piccolo in assoluto
k_attuale = k_anon_counts['Count'].min()
print(f"\nIL NUOVO LIVELLO DI K-ANONYMITY ATTUALE È: k = {k_attuale}")

=== LE CLASSI DI EQUIVALENZA PIÙ PICCOLE (AGGIORNATE) ===
   AGE_GROUP GENDER ZIP_GROUP  Count
7    100-119      F     01***      5
34     80-99      M     01***      6
32     80-99      F     02***      8
11   100-119      M     02***      9
10   100-119      M     01***      9
8    100-119      F     02***      9
35     80-99      M     02***     14
1       0-19      F     01***     15
31     80-99      F     01***     15
30     80-99      F     00***     16

IL NUOVO LIVELLO DI K-ANONYMITY ATTUALE È: k = 5


In [4]:
# 1. Carichiamo i dati sensibili: le condizioni mediche
# Assicurati di avere il file 'conditions.csv' nella stessa cartella
df_conditions = pd.read_csv('conditions.csv')

# 2. Uniamo (Merge) i due dataset
# Nel file patients la colonna è 'Id', nel file conditions la colonna è 'PATIENT'
df_merged = pd.merge(df_patients, df_conditions, left_on='Id', right_on='PATIENT', how='inner')

# 3. IL TOCCO FINALE DI SICUREZZA: Rimuoviamo gli ID di sistema in chiaro!
# L'unico identificativo che ci serve per tenere traccia dello stesso paziente
# nel tempo è il nostro SSN_HASHED.
colonne_da_rimuovere_finali = ['Id', 'PATIENT']
df_final_secure = df_merged.drop(columns=colonne_da_rimuovere_finali)

# 4. VALUTAZIONE UTILITÀ: Testiamo una query tipica
# Domanda: Quali sono le 5 malattie più comuni nelle donne tra i 20 e i 39 anni?
query_donne_giovani = df_final_secure[
    (df_final_secure['GENDER'] == 'F') &
    (df_final_secure['AGE_GROUP'] == '20-39')
]

top_malattie = query_donne_giovani['DESCRIPTION'].value_counts().head(5)

print("=== TOP 5 DIAGNOSI (DONNE 20-39 ANNI) NEL DATASET ANONIMO ===")
print(top_malattie)

=== TOP 5 DIAGNOSI (DONNE 20-39 ANNI) NEL DATASET ANONIMO ===
DESCRIPTION
Full-time employment (finding)    465
Normal pregnancy                  243
Stress (finding)                  215
Viral sinusitis (disorder)        173
Part-time employment (finding)    105
Name: count, dtype: int64


In [5]:
import numpy as np

# 1. Ricarichiamo i dati originali "in chiaro" come Ground Truth per fare il confronto
df_raw_patients = pd.read_csv('patients.csv')
df_raw_patients['BIRTHDATE'] = pd.to_datetime(df_raw_patients['BIRTHDATE'], errors='coerce')
df_raw_patients['AGE'] = anno_corrente - df_raw_patients['BIRTHDATE'].dt.year
df_raw_merged = pd.merge(df_raw_patients, df_conditions, left_on='Id', right_on='PATIENT', how='inner')

# Prepariamo il dataset anonimo calcolando il "Punto Medio" della fascia d'età per le stime
def get_midpoint(age_group):
    if age_group == "Sconosciuta": return np.nan
    parts = age_group.split('-')
    return (int(parts[0]) + int(parts[1])) / 2

df_final_secure['AGE_MIDPOINT'] = df_final_secure['AGE_GROUP'].apply(get_midpoint)

print("=== VALUTAZIONE UTILITÀ: ERRORE RELATIVO SU 3 QUERY ===")

# --- QUERY 1: Età media dei pazienti con Sinusite Virale ---
malattia = "Viral sinusitis (disorder)"
vero_avg_eta = df_raw_merged[df_raw_merged['DESCRIPTION'] == malattia]['AGE'].mean()
anon_avg_eta = df_final_secure[df_final_secure['DESCRIPTION'] == malattia]['AGE_MIDPOINT'].mean()
err_rel_1 = abs(vero_avg_eta - anon_avg_eta) / vero_avg_eta
print(f"\n1. Età media per '{malattia}':")
print(f"   Valore Reale: {vero_avg_eta:.1f} anni | Stima Anonimizzata: {anon_avg_eta:.1f} anni")
print(f"   Errore Relativo: {err_rel_1:.2%}")

# --- QUERY 2: Numero di diagnosi per donne di *esattamente* 30 anni ---
vero_count_30F = len(df_raw_merged[(df_raw_merged['AGE'] == 30) & (df_raw_merged['GENDER'] == 'F')])
# Poiché abbiamo solo la fascia 20-39 (lunga 20 anni), assumiamo una distribuzione uniforme
# e dividiamo il totale della fascia per 20.
count_20_39_F = len(df_final_secure[(df_final_secure['AGE_GROUP'] == '20-39') & (df_final_secure['GENDER'] == 'F')])
stima_count_30F = count_20_39_F / 20
err_rel_2 = abs(vero_count_30F - stima_count_30F) / vero_count_30F if vero_count_30F > 0 else 0
print(f"\n2. Numero diagnosi (Donne, 30 anni esatti):")
print(f"   Valore Reale: {vero_count_30F} | Stima Anonimizzata: {stima_count_30F:.1f}")
print(f"   Errore Relativo: {err_rel_2:.2%}")

# --- QUERY 3: Casi di 'Stress' in una zona specifica ---
# Troviamo il CAP (ZIP) più frequente nel dataset reale
cap_frequente = df_raw_merged['ZIP'].value_counts().idxmax()
vero_stress_cap = len(df_raw_merged[(df_raw_merged['ZIP'] == cap_frequente) & (df_raw_merged['DESCRIPTION'] == 'Stress (finding)')])

# Nel dataset anonimo abbiamo solo le prime due cifre (es. '02***').
cap_anonimo_target = str(int(cap_frequente)).zfill(5)[:2] + "***"
stress_area_anon = len(df_final_secure[(df_final_secure['ZIP_GROUP'] == cap_anonimo_target) & (df_final_secure['DESCRIPTION'] == 'Stress (finding)')])

# Stimiamo dividendo i casi dell'area per il numero di CAP unici originali in quell'area
num_cap_unici = df_raw_merged[df_raw_merged['ZIP'].astype(str).str.zfill(5).str.startswith(cap_anonimo_target[:2])]['ZIP'].nunique()
stima_stress_cap = stress_area_anon / num_cap_unici if num_cap_unici > 0 else 0
err_rel_3 = abs(vero_stress_cap - stima_stress_cap) / vero_stress_cap if vero_stress_cap > 0 else 0

print(f"\n3. Casi di 'Stress' nel CAP specifico {cap_frequente}:")
print(f"   Valore Reale: {vero_stress_cap} | Stima Anonimizzata: {stima_stress_cap:.1f}")
print(f"   Errore Relativo: {err_rel_3:.2%}")

=== VALUTAZIONE UTILITÀ: ERRORE RELATIVO SU 3 QUERY ===

1. Età media per 'Viral sinusitis (disorder)':
   Valore Reale: 53.8 anni | Stima Anonimizzata: 53.8 anni
   Errore Relativo: 0.09%

2. Numero diagnosi (Donne, 30 anni esatti):
   Valore Reale: 105 | Stima Anonimizzata: 137.2
   Errore Relativo: 30.71%

3. Casi di 'Stress' nel CAP specifico 2171.0:
   Valore Reale: 117 | Stima Anonimizzata: 0.0
   Errore Relativo: 100.00%


In [6]:
# FASE 4: Simulazione Rischio di Re-identificazione

# 1. Calcoliamo il Rischio nel Dataset Originale "In Chiaro"
# Quante persone sono uniche (k=1) se incrociamo Età esatta, Sesso e CAP a 5 cifre?
quasi_id_raw = ['AGE', 'GENDER', 'ZIP']
rischio_raw = df_raw_merged.groupby(quasi_id_raw).size().reset_index(name='Count')

pazienti_unici_raw = len(rischio_raw[rischio_raw['Count'] == 1])
totale_pazienti = len(df_raw_merged)
prob_reid_raw = pazienti_unici_raw / totale_pazienti

print("=== SCENARIO 1: RISCHIO NEL DATASET IN CHIARO ===")
print(f"Pazienti completamente unici e vulnerabili (k=1): {pazienti_unici_raw} su {totale_pazienti}")
print(f"Probabilità globale di re-identificazione certa: {prob_reid_raw:.2%}\n")

# 2. Calcoliamo il Rischio nel Dataset Anonimizzato
# Sappiamo già che il nostro k minimo è 5.
pazienti_unici_anon = len(k_anon_counts[k_anon_counts['Count'] == 1])
rischio_massimo_anon = 1 / k_attuale # k_attuale è 5, quindi 1/5 = 20%

print("=== SCENARIO 2: RISCHIO NEL DATASET ANONIMIZZATO ===")
print(f"Pazienti completamente unici e vulnerabili (k=1): {pazienti_unici_anon}")
print(f"Livello k-anonymity minimo garantito: k={k_attuale}")
print(f"Rischio MASSIMO di re-identificazione per il paziente più esposto: {rischio_massimo_anon:.2%}")
print(f"(Nota: Nel dataset in chiaro il rischio per i pazienti più esposti era del 100%)")

=== SCENARIO 1: RISCHIO NEL DATASET IN CHIARO ===
Pazienti completamente unici e vulnerabili (k=1): 7 su 38094
Probabilità globale di re-identificazione certa: 0.02%

=== SCENARIO 2: RISCHIO NEL DATASET ANONIMIZZATO ===
Pazienti completamente unici e vulnerabili (k=1): 0
Livello k-anonymity minimo garantito: k=5
Rischio MASSIMO di re-identificazione per il paziente più esposto: 20.00%
(Nota: Nel dataset in chiaro il rischio per i pazienti più esposti era del 100%)
